In [ ]:
import langchain

In [ ]:
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Command, interrupt
from typing import Annotated, List, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

# Your LLM (e.g., from langchain_openai)
model = llm

# Step 1: Define output structure
class Joke(BaseModel):
    setup: str = Field(description="Question to set up a joke")
    punchline: str = Field(description="Answer to resolve the joke")
    clarification_needed: bool = Field(
        default=False, description="Set to true if the model needs more information to proceed"
    )

# Step 2: Create the output parser
parser = JsonOutputParser(pydantic_object=Joke)

# Step 3: Create a well-instructed prompt template
prompt = PromptTemplate(
    template="""
You are a joke-telling assistant.

Your job is to fill in a structured response that has three fields:
- setup (string)
- punchline (string)
- clarification_needed (boolean)

IMPORTANT RULES:
1. Only write a joke (setup and punchline) if the topic is clearly and explicitly stated in the query.
2. DO NOT GUESS the topic. If you are not sure, DO NOT make assumptions.
3. If the topic is unclear or missing, ask for clarification in the `setup`, leave the `punchline` empty, and set `clarification_needed` to true.

{format_instructions}

Here is the user query: {query}
""",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Step 4: Compose the chain
chain = prompt | model | parser

# Step 5: Now invoke with a real user query (missing topic)
result = chain.invoke({
    "query": "Tell me a joke about that thing we discussed."
})
print(result)


In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
from pydantic import BaseModel, Field

class ThinkerOutput(BaseModel):
    clarification_needed: bool = Field(description="True if clarification is still needed")
    output: str = Field(description="Either a clarification question or the structured project summary")


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [ ]:
class state(TypedDict):
    messages: Annotated[List[str],add_messages]
    clarification_count: int
    clarification_needed: bool

In [ ]:

thinker_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a senior product analyst and technical solution designer.

Your job is to clarify and refine software project ideas through a structured conversation.

Clarification round: {clarification_count}/3

Instructions:
- If the user's idea is unclear, respond ONLY with a **clarification question** that will help you understand the idea better.
- Set `clarification_needed` to true.
- If the idea is sufficiently clear or after 3 rounds, respond with a **structured project summary** covering:
    - Project goals
    - Target users
    - Key components or features
- Set `clarification_needed` to false in this case.

Return your response as a JSON object with exactly two fields:
- `clarification_needed`: true or false
- `output`: a string containing either the clarification question or the project summary

Do NOT output anything outside the JSON.

{format_instructions}

Conversation so far:
{messages}

User's latest input is the last user message in the conversation.
"""
    ),
    MessagesPlaceholder(variable_name="messages"),
])

In [ ]:
# from langchain_core.output_parsers import JsonOutputParser

# parser = JsonOutputParser(pydantic_object=ThinkerOutput)
# # 
# chain = thinker_prompt | llm | parser


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser

# Define the structured output model
class ThinkerOutput(BaseModel):
    clarification_needed: bool = Field(description="True if clarification is still needed")
    output: str = Field(description="Clarification question or project summary")

# Create the chat prompt template with instructions
thinker_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a senior product analyst and technical solution designer.

Your job is to clarify software projects through structured conversation, then deliver a comprehensive project blueprint.

Clarification round: {clarification_count}/3

CLARIFICATION PHASE (clarification_needed = true):
- Ask ONE targeted question about: target users, core functionality, technical needs, or business goals
- Keep questions specific and actionable

FINAL SUMMARY PHASE (clarification_needed = false):
Provide a detailed project blueprint covering:

1. **PROJECT OVERVIEW** - Title, description, objectives, success metrics
2. **TARGET USERS** - User groups, pain points, user journeys  
3. **CORE FEATURES** - Feature breakdown with priorities, user stories
4. **TECHNICAL ARCHITECTURE** - Tech stack, system design, integrations, scalability
5. **IMPLEMENTATION PLAN** - Development phases, MVP scope, timelines, resources
6. **RISKS & MITIGATION** - Technical/business risks and solutions
7. **SUCCESS METRICS** - KPIs and measurement criteria

Make the final summary comprehensive (800-1200 words) and actionable for development planning.

OUTPUT: Return ONLY JSON with two fields:
- `clarification_needed`: true/false
- `output`: clarification question OR detailed project blueprint

{format_instructions}

{messages}
"""
    ),
    MessagesPlaceholder(variable_name="messages"),
])
# Setup the parser
parser = JsonOutputParser(pydantic_object=ThinkerOutput)

# Compose the full chain: prompt -> LLM model -> parser
chain = thinker_prompt | llm | parser

# Initialize conversation state
state = {
    "messages": [],
    "clarification_count": 0,
    "clarification_needed": True,
}

def run_thinker(user_input: str):
    # Append new user input to conversation messages
    state["messages"].append({"role": "user", "content": user_input})

    # Call the chain with current state
    response = chain.invoke({
        "messages": state["messages"],
        "clarification_count": state["clarification_count"],
        "format_instructions": parser.get_format_instructions(),
    })
    response = ThinkerOutput(**response)
    # Update state based on LLM response
    if response.clarification_needed :
        state["clarification_needed"] = True
        state["clarification_count"] += 1
        # Add assistant's clarification question to messages
        state["messages"].append({"role": "assistant", "content": response.output})
    else:
        state["clarification_needed"] = False
        # Add the final structured summary to messages
        state["messages"].append({"role": "assistant", "content": response.output})

    return response
    

# Example usage:
user_idea = input("Enter the project idea : ")
result = run_thinker(user_idea)
print(result)


In [ ]:
user_idea = input("Enter the clarification_question ")
result = run_thinker(user_idea)
print(result.output)
print(state["messages"])
user_idea = input("Enter the clarification_question ")
result = run_thinker(user_idea)
print(result.output)
print(state["messages"])
user_idea = input("Enter the clarification_question ")
result = run_thinker(user_idea)
print(result.output)
print(state["messages"])

In [12]:
output = """
    ## Library Management System Website: Project Blueprint

**1. PROJECT OVERVIEW**

*   **Title:** Library Management System Website
*   **Description:** A comprehensive web-based platform to manage library resources, user accounts, and library operations. This system will cater to various user roles, including librarians, students, and faculty, providing role-specific access and functionalities.
*   **Objectives:**
    *   Enable efficient catalog searching and browsing.
    *   Facilitate user account management (registration, login, profile updates).
    *   Streamline the borrowing and returning process.
    *   Provide inventory tracking and management tools for librarians.
    *   Generate reports on library usage and resource availability.
    *   Offer a user-friendly and accessible interface for all user types.
*   **Success Metrics:**
    *   Increased user engagement (measured by website visits, searches, and borrowing activity).
    *   Improved efficiency in library operations (reduced checkout/check-in times, faster inventory management).
    *   High user satisfaction (measured by surveys and feedback).
    *   Reduced administrative overhead (automated tasks, online reporting).

**2. TARGET USERS**

*   **User Groups:**
    *   **Students:** Primary users for searching the catalog, borrowing and returning books, managing their accounts, and accessing digital resources.
    *   **Faculty:** Similar access to students, with potential for reserving materials for courses and managing research resources.
    *   **Librarians:** Administrative users responsible for managing the catalog, user accounts, inventory, borrowing/returning processes, and generating reports.
*   **Pain Points:**
    *   **Students/Faculty:** Difficulty finding desired books, long queues for borrowing/returning, limited access to digital resources, inconvenient account management.
    *   **Librarians:** Time-consuming manual processes, difficulty tracking inventory, inefficient reporting, challenges in managing user accounts.
*   **User Journeys:**
    *   **Student:**
        1.  Logs in to the website.
        2.  Searches for a book by title, author, or keyword.
        3.  Views book details (availability, summary, reviews).
        4.  Checks out the book (if available).
        5.  Receives email notifications about due dates.
        6.  Returns the book.
        7.  Manages their account profile (updates contact information).
    *   **Librarian:**
        1.  Logs in to the admin panel.
        2.  Adds a new book to the catalog.
        3.  Updates the availability of a book.
        4.  Manages user accounts (creates, edits, deactivates).
        5.  Generates a report on overdue books.
        6.  Processes book returns and check-ins.

**3. CORE FEATURES**

*   **Feature Breakdown:**
    *   **Catalog Search & Browsing (High):** Allows users to search for books and other resources by title, author, ISBN, keyword, etc. Includes advanced search filters and browsing by category.
    *   **User Account Management (High):** Enables users to register, login, manage their profiles, view borrowing history, and receive notifications.
    *   **Borrowing & Returning (High):** Facilitates the borrowing and returning of books, including due date tracking, renewals, and late fee management.
    *   **Inventory Management (High):** Allows librarians to add, edit, and remove books from the catalog, track inventory levels, and manage book locations.
    *   **Reporting & Analytics (Medium):** Generates reports on library usage, overdue books, popular resources, and other relevant metrics.
    *   **Digital Resource Access (Medium):** Provides access to e-books, online journals, and other digital resources.
    *   **Reservation System (Medium):** Allows users to reserve books that are currently checked out.
    *   **Admin Panel (High):** A dedicated interface for librarians to manage all aspects of the system.
*   **User Stories:**
    *   As a student, I want to be able to easily search for books so that I can find the resources I need for my studies.
    *   As a librarian, I want to be able to quickly add new books to the catalog so that the system is always up-to-date.
    *   As a librarian, I want to be able to generate reports on overdue books so that I can manage outstanding loans effectively.
    *   As a student, I want to receive email notifications about upcoming due dates so that I can avoid late fees.

**4. TECHNICAL ARCHITECTURE**

*   **Tech Stack:**
    *   **Frontend:** React (for a dynamic and responsive user interface).
    *   **Backend:** Node.js with Express.js (for a scalable and efficient server-side application).
    *   **Database:** PostgreSQL (for reliable data storage and management).
    *   **Hosting:** AWS or Google Cloud Platform (for scalability and availability).
*   **System Design:**
    *   Three-tier architecture: Frontend (React), Backend (Node.js/Express.js), and Database (PostgreSQL).
    *   RESTful API for communication between the frontend and backend.
    *   Authentication and authorization using JWT (JSON Web Tokens).
*   **Integrations:**
    *   Email service (e.g., SendGrid, Mailgun) for sending notifications.
    *   Payment gateway (e.g., Stripe, PayPal) for online fee payments (optional).
    *   Library API (if available) for external data integration.
*   **Scalability:**
    *   Horizontal scaling of the backend servers.
    *   Database replication and sharding.
    *   Caching mechanisms (e.g., Redis) to improve performance.

**5. IMPLEMENTATION PLAN**

*   **Development Phases:**
    1.  **Phase 1: MVP (Minimum Viable Product)**
        *   Catalog search and browsing.
        *   User account management (registration, login, profile).
        *   Borrowing and returning functionality.
        *   Admin panel for basic inventory management.
    2.  **Phase 2: Enhanced Features**
        *   Reporting and analytics.
        *   Reservation system.
        *   Digital resource access.
        *   Advanced search filters.
    3.  **Phase 3: Integrations & Optimization**
        *   Email service integration.
        *   Payment gateway integration (optional).
        *   Performance optimization.
        *   Accessibility improvements.
*   **MVP Scope:**
    *   Focus on core functionalities required for basic library operations.
    *   Simple and intuitive user interface.
    *   Robust security measures.
*   **Timelines:**
    *   Phase 1 (MVP): 8-12 weeks.
    *   Phase 2: 6-8 weeks.
    *   Phase 3: 4-6 weeks.
*   **Resources:**
    *   Frontend developers (2).
    *   Backend developers (2).
    *   Database administrator (1).
    *   UI/UX designer (1).
    *   Project manager (1).
    *   QA tester (1).

**6. RISKS & MITIGATION**

*   **Technical Risks:**
    *   **Scalability issues:** Implement caching and database optimization techniques.
    *   **Security vulnerabilities:** Conduct regular security audits and penetration testing.
    *   **Integration c
    
    
    hallenges:** Thoroughly test integrations with external services.
*   **Business Risks:**
    *   **User adoption:** Conduct user training and provide ongoing support.
    *   **Data migration:** Develop a comprehensive data migration plan.
    *   **Scope creep:** Clearly define project scope and manage change requests effectively.

**7. SUCCESS METRICS**

*   **Key Performance Indicators (KPIs):**
    *   Number of registered users.
    *   Website traffic (page views, unique visitors).
    *   Search query volume.
    *   Borrowing and returning rates.
    *   User satisfaction scores.
    *   Time spent on website.
    *   Number of overdue books.
*   **Measurement Criteria:**
    *   Track website traffic using Google Analytics.
    *   Monitor user activity through database queries.
    *   Conduct user surveys to gather feedback.
    *   Analyze system logs to identify performance bottlenecks.
    *   Regularly review KPIs to assess project success and identify areas for improvement.
"""

In [ ]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv  
from typing import Any, Dict

load_dotenv()

class DirSchema(BaseModel):
    project_name: str = Field(description="Name of the project")
    structure: Dict[str, Any] = Field(description="Directory structure in JSON format")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")


directory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """ 
            You are a senior software architect and project manager.
            Your job is to create a comprehensive directory and its file structure for software projects.
            You will receive a project name and a detailed project description and tech stack.
            Your output should be a JSON object with two fields:
            - `project_name`: the name of the project
            - `structure`: a JSON object representing the directory structure
            The structure should be well-organized, modular, and suitable for a software project of this type.

            Note : use the standard naming conventions for directories and files.
            The output should be a valid JSON object with the following format:

        """
    ),
        ("user", "{Project_description}")
])

parser = JsonOutputParser(pydantic_object=DirSchema)

# Compose the full chain: prompt -> LLM model -> parser
chain = directory_prompt | llm | parser

# Example usage
result = chain.invoke({
    "Project_description": output
})
print(result)

In [ ]:
import json
with open("dir",'w') as fp:
    json.dump(result,fp)
# result

In [ ]:
import os
import sys

def create_structure(base_path, structure_dict):
    """Recursively creates directories and files based on the structure dictionary."""
    for name, content in structure_dict.items():
        item_path = os.path.join(base_path, name)

        if isinstance(content, dict):
            # It's a directory
            print(f"Creating directory: {item_path}")
            os.makedirs(item_path, exist_ok=True) # exist_ok=True prevents error if dir already exists
            # Recurse into this new directory
            create_structure(item_path, content)
        elif isinstance(content, str):
            # It's a file (value is an empty string)
            print(f"Creating file: {item_path}")
            # Create an empty file
            try:
                with open(item_path, 'w') as f:
                    f.write(content) # Write the content (empty string in this case)
            except IOError as e:
                print(f"Error creating file {item_path}: {e}", file=sys.stderr)

        else:
            print(f"Warning: Unknown type for item '{name}' at path '{base_path}'. Skipping.", file=sys.stderr)


if __name__ == "__main__":
    # 1. Parse the JSON string
    try:
        project_data = result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}", file=sys.stderr)
        sys.exit(1)

    # 2. Get the project name and the root structure
    project_name = project_data.get("project_name", "my_project") # Default name if not specified
    root_structure = project_data.get("structure", {}).get("root", {})

    if not root_structure:
        print("Error: JSON structure does not contain a 'structure.root' key.", file=sys.stderr)
        sys.exit(1)

    # 3. Create the base project directory
    print(f"Creating base project directory: temp")
    os.makedirs("temp", exist_ok=True)

    # 4. Start the recursive creation process from the root
    create_structure("temp", root_structure)

    print("\nProject structure created successfully!")
    print(f"Look for the '{project_name}' directory.")

In [ ]:
import os
import requests
import json
from pydantic import BaseModel, Field, validator
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain.tools import StructuredTool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from dotenv import load_dotenv
from typing import Any, Dict, List

load_dotenv()

# --- Pydantic Schemas ---
class DirSchema(BaseModel):
    project_name: str = Field(description="Name of the project")
    structure: Dict[str, Any] = Field(description="Directory structure in JSON format")

class GitHubSearchToolInput(BaseModel):
    query: str = Field(description="The search query for GitHub repositories.")
    count: int = Field(default=3, description="Number of repositories to fetch. Max 5.")

    @validator('count')
    def count_must_be_positive_and_limited(cls, v):
        if v <= 0 or v > 5:
            raise ValueError("Count must be between 1 and 5")
        return v

# --- LLM Initialization ---
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", temperature=0) # Using a specific model

# --- GitHub Search Functionality (Helper functions) ---
GITHUB_API_URL = "https://api.github.com"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

def _search_github_repositories_raw(query: str, count: int = 3) -> List[Dict[str, str]]:
    headers = {"Accept": "application/vnd.github.v3+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"token {GITHUB_TOKEN}"
    search_url = f"{GITHUB_API_URL}/search/repositories"
    params = {"q": query, "sort": "stars", "order": "desc", "per_page": count}
    try:
        print(search_url,end="\n\n\n\n")
        response = requests.get(search_url, headers=headers, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        return [
            {
                "name": item.get("full_name"),
                "description": item.get("description", "No description provided."),
                "url": item.get("html_url")
            } for item in data.get("items", [])
        ]
    except requests.exceptions.RequestException as e:
        print(f"Error searching GitHub: {e}")
        return []

def _format_github_results_for_prompt(results: List[Dict[str, str]]) -> str:
    if not results:
        return "No relevant GitHub repositories found or GitHub search failed."
    formatted_string = "Found the following potentially relevant GitHub repositories:\n"
    for i, repo in enumerate(results):
        formatted_string += f"{i+1}. Name: {repo['name']}\n"
        formatted_string += f"   Description: {repo['description']}\n"
        formatted_string += f"   URL: {repo['url']}\n\n"
    return formatted_string.strip()

def github_search_tool_function(query: str, count: int = 3) -> str:
    """
    Searches GitHub for repositories based on a query and returns formatted results.
    Use this tool to find example projects, boilerplates, or common structures.
    """
    print(f"---> GitHub Tool Called with Query: '{query}', Count: {count}")
    raw_results = _search_github_repositories_raw(query, count)
    formatted_results = _format_github_results_for_prompt(raw_results)
    print(f"---> GitHub Tool Results:\n{formatted_results}")
    return formatted_results

# --- Create Langchain Tool ---
github_search_tool = StructuredTool.from_function(
    func=github_search_tool_function,
    name="GitHubRepositorySearch",
    description="Searches GitHub for repositories based on a query and returns formatted results of names, descriptions, and URLs. Use this to find example projects, boilerplates, or common structures relevant to the user's request.",
    args_schema=GitHubSearchToolInput
)
tools = [github_search_tool]

# --- Agent for GitHub Info Retrieval ---
# Define a clear system message for the agent's role and instructions.
AGENT_SYSTEM_MESSAGE = """
You are a helpful assistant. Your task is to find relevant GitHub repositories based on the user's project details.
1.  Analyze the user's input, which contains the project description and technical stack.
2.  Formulate one or two concise and effective search queries for GitHub based on these details.
3.  Use the 'GitHubRepositorySearch' tool with your best query.
4.  Present the information returned by the tool as your final answer.
If the user's request is unclear or lacks detail for a good search, state that you need more information.
Do not make up repository information if the tool returns nothing.
"""

# The agent prompt should typically include an "input" placeholder for the user's query
# and an "agent_scratchpad" for intermediate steps.
agent_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", AGENT_SYSTEM_MESSAGE),
        # This "human" message will be populated by the 'input' key in agent_executor.invoke
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm_with_tools = llm.bind_tools(tools)
github_info_agent = create_tool_calling_agent(
    llm=llm_with_tools,
    tools=tools,
    prompt=agent_prompt
)
github_agent_executor = AgentExecutor(agent=github_info_agent, tools=tools, verbose=True)

# --- Enhanced Directory Prompt (remains similar) ---
directory_prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """
            You are a senior software architect and project manager.
            Your job is to create a comprehensive directory and file structure for software projects.
            You will receive a project name, a detailed project description, tech stack, and information
            from existing GitHub repositories that seem relevant.

            Consider the GitHub information to understand common patterns and best practices for similar projects,
            but prioritize creating a structure that is well-organized, modular, and suitable for the specific project described.
            Don't just copy a structure from GitHub; synthesize and adapt.

            Your output should be a JSON object with two fields:
            - `project_name`: the name of the project
            - `structure`: a JSON object representing the directory structure and dont write anything to at last json file

            Use standard naming conventions for directories and files. The output should be a valid JSON object.

            Relevant GitHub Repository Information:
            {github_info}
        """
    ),
    ("user", "Project Name: {project_name}\n\nProject Description:\n{Project_description}\n\nTech Stack:\n{Tech_stack}")
])

# --- Output Parser (remains similar) ---
parser = JsonOutputParser(pydantic_object=DirSchema)

# --- Combined Scaffolding Chain ---
def run_agent_and_prepare_output(initial_input_dict: Dict) -> Dict:
    """
    Runs the GitHub agent and prepares the dictionary for the next step.
    """
    # Combine project description and tech stack into a single 'input' string for the agent
    agent_input_str = (
        f"Project Description: {initial_input_dict['Project_description']}\n\n"
        f"Technical Stack: {initial_input_dict.get('Tech_stack', 'Not specified')}"
    )
    print(f"\n---> Running GitHub Agent with combined input string:\n{agent_input_str[:200]}...") # Print first 200 chars

    # The agent_executor expects an 'input' key that matches the "{input}" in the agent_prompt
    agent_response = github_agent_executor.invoke({
        "input": agent_input_str
    })

    # The final answer from the agent is usually in the 'output' key
    github_info = agent_response.get("output", "Agent did not produce GitHub info. No repositories found or an error occurred.")
    print(f"---> Agent Output (github_info):\n{github_info}")

    # Return a dictionary including the original inputs and the new github_info
    return {
        "project_name": initial_input_dict["project_name"],
        "Project_description": initial_input_dict["Project_description"],
        "Tech_stack": initial_input_dict["Tech_stack"],
        "github_info": github_info
    }

scapefolding_chain = (
    RunnableLambda(run_agent_and_prepare_output) # This runs the agent and formats its output
    | directory_prompt_template
    | llm # This is the second LLM call for directory structuring
    | parser
)

# --- Example Usage ---
if __name__ == "__main__":
    project_input_main = { # Renamed to avoid conflict if you had 'project_input' before
        "project_name": "AI Powered Blog Generator",
        "Project_description": (
            "A web application that uses generative AI (like GPT models) to help users create blog posts. "
            "Users can provide a topic, keywords, and desired tone, and the system will generate a draft. "
            "It should have a frontend for user interaction, a backend API to handle requests and interact with the AI model, "
            "and a database to store user accounts and generated content."
        ),
        "Tech_stack": "Python (FastAPI) for backend, React for frontend, PostgreSQL for database, Docker for containerization. Using OpenAI API for generation."
    }

    print("\n--- Generating Project Structure (with Agent-based GitHub insights) ---")
    try:
        result = scapefolding_chain.invoke(project_input_main)
        print("\n\n--- Final Project Structure Output ---")
        print(json.dumps(result, indent=2))
    except Exception as e:
        print(f"An error occurred during chain execution: {e}")
        import traceback
        traceback.print_exc()

    print("\n--- Example with minimal input (no explicit tech stack) ---")
    minimal_project_input = {
        "project_name": "Simple Task Manager CLI",
        "Project_description": "A command-line tool to manage daily tasks. Add, remove, list tasks. To be written in Python.",
        "Tech_stack": "Python" # Good to provide if known
    }
    try:
        result_minimal = scapefolding_chain.invoke(minimal_project_input)
        print("\n\n--- Final Project Structure Output (Minimal Input) ---")
        print(json.dumps(result_minimal, indent=2))
    except Exception as e:
        print(f"An error occurred during chain execution (minimal input): {e}")
        import traceback
        traceback.print_exc()

In [ ]:
from langchain.tools import tool
import os

@tool("create_file")
def create_file(path: str, filename: str) -> bool:
    """Creates a file at the given path with the given filename."""
    if os.path.exists(path) and os.path.isdir(path):
        filepath = os.path.join(path, filename)
        with open(filepath, 'w') as f:
            pass
        return True
    return False

@tool("create_dir")
def create_dir(path: str, foldername: str) -> bool:
    """Creates a directory at the given path with the given folder name."""
    if os.path.exists(path) and os.path.isdir(path):
        folderpath = os.path.join(path, foldername)
        os.mkdir(folderpath)
        return True
    return False
 
# create_dir.invoke({"path": "./", "foldername": "hello"})

# create_file.invoke({"path": "./hello", "filename": "hello.ts"})

# github

In [ ]:
import requests
from urllib.parse import urlparse

def parse_github_url(url):
    parts = urlparse(url).path.strip('/').split('/')
    if len(parts) >= 2:
        return parts[0], parts[1]
    raise ValueError(f"Invalid GitHub URL: {url}")

def get_default_branch(owner, repo, token=None):
    headers = {'Authorization': f'token {token}'} if token else {}
    url = f"https://api.github.com/repos/{owner}/{repo}"
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    return r.json()['default_branch']

def get_repo_structure(owner, repo, branch, token=None):
    headers = {'Authorization': f'token {token}'} if token else {}
    
    # Get tree SHA
    sha_url = f"https://api.github.com/repos/{owner}/{repo}/branches/{branch}"
    sha_data = requests.get(sha_url, headers=headers).json()
    tree_sha = sha_data['commit']['commit']['tree']['sha']

    # Get full tree
    tree_url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{tree_sha}?recursive=1"
    tree_data = requests.get(tree_url, headers=headers).json()

    if 'tree' not in tree_data:
        return {}

    tree = tree_data['tree']
    structure = {}
    for item in tree:
        parts = item['path'].split('/')
        current = structure
        for part in parts[:-1]:
            current = current.setdefault(part, {})
        if item['type'] == 'tree':
            current.setdefault(parts[-1], {})
        else:
            current[parts[-1]] = "file"
    return structure

def build_tree_string(structure, indent=0):
    lines = []
    for key, value in sorted(structure.items()):
        lines.append('    ' * indent + '└── ' + key)
        if isinstance(value, dict):
            lines.extend(build_tree_string(value, indent + 1))
    return lines


def get_repos_tree_from_urls(urls, token=None):
    all_projects = []

    for url in urls:
        try:
            owner, repo = parse_github_url(url)
            branch = get_default_branch(owner, repo, token)
            structure = get_repo_structure(owner, repo, branch, token)
            tree_str = "\n".join(build_tree_string(structure))
            all_projects.append({
                "owner": owner,
                "project": repo,
                "tree": tree_str
            })
        except Exception as e:
            print(f"Error processing {url}: {e}")
            continue

    return all_projects


In [ ]:
urls = [
    "https://github.com/chartjs/chartjs-chart-financial",
    "https://github.com/okTurtles/group-income",
    "https://github.com/Emurgo/yoroi-frontend",
    "https://github.com/janlukasschroeder/realtime-newsapi"
]

projects = get_repos_tree_from_urls(urls)

from pprint import pprint
pprint(projects)
